# OGC API Processes — Execution, Async Jobs, Dismiss

Demonstrates OGC API — Processes (Parts 1 + 4) against DynaStore.

## What this notebook covers

1. List available processes and inspect a process schema.
2. Synchronous execution (`Prefer: wait=30`) → HTTP 200.
3. Async execution (`Prefer: respond-async`) → HTTP **201** + `Location` header.
4. Poll job with exponential backoff until terminal status.
5. Retrieve job results.
6. Dismiss job (`DELETE /processes/jobs/{id}`) → HTTP 200 `StatusInfo`.

**Note on HTTP status codes** — DynaStore's processes service uses **201** for
async job creation (see `processes_service.py:744`), not 202.  The `Location`
header in the 201 response points to the job status URL.

**Env vars:** `DYNASTORE_BASE_URL` (default `http://localhost:8080`),
`DYNASTORE_SYSADMIN_TOKEN` or `DYNASTORE_TOKEN`,
`PROCESS_ID` (default: first registered process).

**Routes verified against** `processes_service.py`:
- `GET /processes/processes` → 200
- `GET /processes/processes/{id}` → 200
- `POST /processes/processes/{id}/execution` → 200 (sync) or 201 (async)
- `GET /processes/jobs/{id}` → 200
- `GET /processes/jobs/{id}/results` → 200
- `DELETE /processes/jobs/{id}` → 200 StatusInfo

In [ ]:
import os
import json
import time
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)

if not TOKEN:
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                TOKEN = _r.json().get("access_token", "")
        except Exception:
            pass

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client = httpx.Client(base_url=BASE, headers=HEADERS, timeout=60.0)

PROCESS_ID = os.environ.get("PROCESS_ID", "geometry-validator")
job_id = None  # populated by async execution cell

print(f"Base URL   : {BASE}")
print(f"PROCESS_ID : {PROCESS_ID}")

## 1. List processes and auto-discover PROCESS_ID

In [ ]:
r = client.get("/processes/processes")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text[:300]}"

body = r.json()
processes = body.get("processes", body) if isinstance(body, dict) else body
if isinstance(processes, dict):
    processes = list(processes.values())

print(f"Registered processes ({len(processes)}):")
available_ids = []
for proc in processes:
    pid = proc.get("id", proc.get("name", "?"))
    title = proc.get("title", proc.get("summary", ""))
    print(f"  {pid}: {title}")
    available_ids.append(pid)

if PROCESS_ID not in available_ids and available_ids:
    print(f"NOTE: '{PROCESS_ID}' not registered; falling back to '{available_ids[0]}'")
    PROCESS_ID = available_ids[0]

SKIP = PROCESS_ID not in available_ids
if SKIP:
    print("SKIP: No processes registered — remaining cells will be skipped.")
else:
    print(f"Using PROCESS_ID: {PROCESS_ID}")

## 2. Inspect process schema

In [ ]:
if SKIP:
    print("SKIP")
else:
    r = client.get(f"/processes/processes/{PROCESS_ID}")
    assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text[:300]}"
    proc_schema = r.json()
    print(f"Process  : {proc_schema.get('id')} — {proc_schema.get('title', '')}")
    print("Inputs:")
    for name, defn in proc_schema.get("inputs", {}).items():
        schema = defn.get("schema", {})
        print(f"  {name}: type={schema.get('type', '?')}")
    print("Outputs:")
    for name, defn in proc_schema.get("outputs", {}).items():
        schema = defn.get("schema", {})
        print(f"  {name}: type={schema.get('type', '?')}")

    # Build a minimal valid execution payload from the schema
    EXEC_PAYLOAD = {
        "inputs": {
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]],
            }
        },
        "outputs": {"report": {"transmissionMode": "value"}},
    }
    print("\nExecution payload ready.")

## 3. Synchronous execution (`Prefer: wait=30`)

When the process completes within the wait window the server returns
**HTTP 200** with the result directly.

In [ ]:
if SKIP:
    print("SKIP")
else:
    r = client.post(
        f"/processes/processes/{PROCESS_ID}/execution",
        content=json.dumps(EXEC_PAYLOAD),
        headers={**HEADERS, "Prefer": "wait=30"},
    )
    # 200 = synchronous result; 201 = async accepted (server could not complete within wait window)
    assert r.status_code in (200, 201), (
        f"Expected 200 (sync result) or 201 (async fallback); got {r.status_code}: {r.text[:300]}"
    )
    result = r.json()
    print(f"HTTP {r.status_code}  status={result.get('status', '(direct result)')}")

    if r.status_code == 200:
        assert result.get("status") == "successful" or "outputs" in result or "results" in result, (
            f"Sync result must carry status=successful or outputs; got keys: {list(result.keys())}"
        )
        print("Synchronous execution successful.")
    else:
        # Async fallback — save job_id for later cells
        job_id = result.get("jobID", result.get("job_id", ""))
        location = r.headers.get("Location", "(missing)")
        assert job_id, f"201 returned but no jobID in response: {result}"
        assert location != "(missing)", "201 must include a Location header"
        print(f"Async fallback: job_id={job_id}  Location={location}")

## 4. Async execution (`Prefer: respond-async`)

Forces async mode.  The server returns **HTTP 201** with a `StatusInfo`
body and a `Location` header pointing to the job status URL.

**DynaStore uses 201, not 202** — see `processes_service.py:744`.

In [ ]:
if SKIP:
    print("SKIP")
else:
    r = client.post(
        f"/processes/processes/{PROCESS_ID}/execution",
        content=json.dumps(EXEC_PAYLOAD),
        headers={**HEADERS, "Prefer": "respond-async"},
    )
    # DynaStore returns 201 for async job creation, not 202
    assert r.status_code == 201, (
        f"Expected 201 with Prefer: respond-async; got {r.status_code}: {r.text[:300]}"
    )

    receipt = r.json()
    assert receipt.get("status") == "accepted", (
        f"Expected status='accepted' in 201 response; got '{receipt.get('status')}'"
    )

    location = r.headers.get("Location", "")
    assert location, f"201 must include Location header; headers: {dict(r.headers)}"

    job_id = receipt.get("jobID", receipt.get("job_id", ""))
    assert job_id, f"No jobID in 201 response: {receipt}"

    print(f"HTTP 201  status=accepted  job_id={job_id}")
    print(f"Location : {location}")

## 5. Poll job with exponential backoff

In [ ]:
if SKIP or not job_id:
    print("SKIP: no job to poll.")
    job_status = None
else:
    job_status = "accepted"
    for attempt in range(20):
        r = client.get(f"/processes/jobs/{job_id}")
        assert r.status_code == 200, (
            f"Expected 200 polling job; got {r.status_code}: {r.text[:300]}"
        )
        job_status = r.json().get("status", "")
        print(f"  Attempt {attempt + 1}: status={job_status}")
        if job_status in ("successful", "failed", "dismissed"):
            break
        time.sleep(min(2 ** attempt, 30))

    print(f"Terminal status: {job_status}")

## 6. Retrieve job results

In [ ]:
if SKIP or not job_id or job_status != "successful":
    print(f"SKIP: job_id={job_id}  job_status={job_status}")
else:
    r = client.get(f"/processes/jobs/{job_id}/results")
    assert r.status_code == 200, f"Expected 200; got {r.status_code}: {r.text[:300]}"
    results = r.json()
    # If the job failed the body is an OGC exception report (has 'type' with 'exception')
    if isinstance(results, dict) and "exception" in results.get("type", "").lower():
        raise RuntimeError(f"Job {job_id} failed: {results.get('detail', results)}")
    print("Results:")
    print(json.dumps(results, indent=2)[:800])

## Teardown — dismiss job

`DELETE /processes/jobs/{id}` returns **200 StatusInfo** with
`status=dismissed` (see `processes_service.py:1000–1016`).

In [ ]:
if job_id:
    r = client.delete(f"/processes/jobs/{job_id}")
    assert r.status_code == 200, (
        f"Expected 200 StatusInfo on dismiss; got {r.status_code}: {r.text[:300]}"
    )
    status_info = r.json()
    assert status_info.get("status") == "dismissed", (
        f"Expected status='dismissed'; got '{status_info.get('status')}'"
    )
    print(f"Job {job_id} dismissed. StatusInfo.status={status_info.get('status')}")
else:
    print("No job to dismiss.")

client.close()
print("Teardown complete.")